# FRAUD DETECTION 

A classic problem in financial environments. We will take a look at classifying transitions into fraud and not fraud using scikit-learn package.

This notebook will be using a synthetic database found on kaggle for educational purposes because we all need practice :-).   
The data can be found here: https://www.kaggle.com/datasets/jayjoshi37/digital-payment-fraud-detection/data

I will break this proces down into the follwoing steps:

1. Data Exploration 
2. Data prepartion and pre-processing (including feature engineering)
3. Modelling 
4. Evaluation and testing 

We have the following fields within our dataset: 

transaction_id - identifier to each transaction.   
user_id - identifier for each user.   
transaction_amount - amount for each transaction.   
transaction_type - how funds were exchanged e.g "payment" or "bank transfer".   
payment_mode - wallet, card, UPI etc.   
device_type - device transaction was made from e.g iOS.   
device_location - location of the device used to make transaction.   
account_age_days - age of the account.     
transaction_hour - time of transaction in 24 hour notation.    
previous_failed_attempts - if there were previous attempts to make fraudulent transactions.   
avg_transaction_amount - avg amount each account usually makes.   
is_international - is the trasnaction international.    
ip_risk_score - a numerical value  that quantifies the likelihood an IP address is involved in malicious activity, such as fraud, spam, or cyberattacks. 
login_attempts_last_24h - number of login attempts to the account in the last 24 hours.   
fraud_label- is the transaction fraud or not.


## Data Exploration 

In [1]:
# First we need to start by acquiring all of our dependecies 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
# Now we will load our data into our notebook for modelling:
current_dir = Path.cwd()
print(current_dir)              # this directory is 1 level too low in the project structure
parent_dir = current_dir.parent # go up 1 level in the structure 
print(parent_dir)               # now we are in the parent directory that we can now use to naviagte into the dataset 
file_path = parent_dir/ "dataset" / "Digital_Payment_Fraud_Detection_Dataset.csv" 
print(file_path)                # now we are in the correct bracnh of the tree where our dataset is

fraud = pd.read_csv(file_path)  # IT WORKED!!!


/Users/leta/Desktop/Data Science Career/Python/Python Projects/FraudDetection/modelling
/Users/leta/Desktop/Data Science Career/Python/Python Projects/FraudDetection
/Users/leta/Desktop/Data Science Career/Python/Python Projects/FraudDetection/dataset/Digital_Payment_Fraud_Detection_Dataset.csv


In [ ]:
fraud.shape # dimensions of the dataframe

(7500, 15)

In [ ]:
# Let us take a look into the data we have 
fraud.head()

,transaction_id,user_id,transaction_amount,transaction_type,payment_mode,device_type,device_location,account_age_days,transaction_hour,previous_failed_attempts,avg_transaction_amount,is_international,ip_risk_score,login_attempts_last_24h,fraud_label
0,T1,U3756,18758.28,Transfer,UPI,Web,Hyderabad,895,14,1,25535.84,0,0.718,4,0
1,T2,U7899,47538.18,Payment,Wallet,iOS,Hyderabad,918,21,0,3955.85,0,0.525,9,0
2,T3,U1765,36613.10,Payment,Card,Android,Chennai,1506,8,4,22727.71,0,0.985,9,0
3,T4,U8850,29952.99,Payment,Wallet,iOS,Chennai,800,1,3,18095.89,0,0.797,2,0
4,T5,U9049,7843.13,Payment,UPI,Web,Delhi,301,4,1,9317.49,1,0.468,1,0


In [ ]:
# In this dataset we have mostly numerical variables with a few catgeroical variables.
# Let us look at what fields we have
fraud.keys() #column names of us data


Index(['transaction_id', 'user_id', 'transaction_amount', 'transaction_type',
       'payment_mode', 'device_type', 'device_location', 'account_age_days',
       'transaction_hour', 'previous_failed_attempts',
       'avg_transaction_amount', 'is_international', 'ip_risk_score',
       'login_attempts_last_24h', 'fraud_label'],
      dtype='str')

In [6]:
#What unique labels do we have in the dataset 
fraud.nunique()
# The fields with the highest unique values are the ID fields, transaction amounts, account ages, avgerage transaction amounts, ip risk scores 
# ID fields are just unqiue ientifiers but have no bearing on prediction therefore we will drop these fields especially since they have many unique values.  
# We will plore some more of these fields to see there bearing on the predcitve power of our models using exploratory data analysis and data mining.

transaction_id              7500
user_id                     5106
transaction_amount          7499
transaction_type               3
payment_mode                   4
device_type                    3
device_location                5
account_age_days            1943
transaction_hour              24
previous_failed_attempts       5
avg_transaction_amount      7498
is_international               2
ip_risk_score               1000
login_attempts_last_24h        9
fraud_label                    2
dtype: int64

In [ ]:
# Finally let us see of the data is balanced or not 
fraud["fraud_label"].value_counts(normalize=True) #count all fraud labels and get proportion

# Fraud - 6.52% of observations    
# Non fraud - 93.28% of observations 

# Our data is very skewed towards non fraud transactions.
# This will infrom our metric to measure perfomrance (probably balanced accuarcy, recall and precision) 
# as well as how we stratify the data when partitioning into test and train sets. 


fraud_label
0    0.9348
1    0.0652
Name: proportion, dtype: float64

Our variables with categorical information being: transaction_type, payment_mode, device_type and device_location do not have too many unqiue values.  
This infroms what methods we can use to deal with these values for classification models that only use numerical data.  
- A possible solution would be to use dummy variables. 

##  DATA PREPARATION AND PRE-PROCESSING 


FEATURE ENGINEERING   
We will now create features that improve the performance of our models based on the EDA we conducted in R and the insights we gathered. 

In [ ]:
# Now we will take partition the data into dependent and indepenedt variables. 
x =fraud[['user_id', 'transaction_amount', 'device_location', 'account_age_days',
       'transaction_hour', 'previous_failed_attempts', 'payment_mode', 'device_type', 'transaction_type',
       'avg_transaction_amount', 'is_international', 'ip_risk_score',
       'login_attempts_last_24h']]
y = fraud[['fraud_label']]                               # subsetting our predicted variable 

In [226]:
# Partitioning our data 
# The data will be partitioned into tran and test splits at a 70/30% proportion 

from sklearn.model_selection import train_test_split # package to spilt our data 

x_train, x_test, y_train, y_test = train_test_split(x,y,              # our dependent and independent variables to be used
                                                    stratify = y,     # keeping proportion of fraud and not fraud equal in the the train and test sets
                                                    random_state=42,  # setting a seed for reproducibility 
                                                    test_size=0.3)    # 30% test size


In [227]:
x_train.keys() 
# Making sure we have all of the columns necessary in our trainng set with no data leakage.

Index(['user_id', 'transaction_amount', 'device_location', 'account_age_days',
       'transaction_hour', 'previous_failed_attempts', 'payment_mode',
       'device_type', 'transaction_type', 'avg_transaction_amount',
       'is_international', 'ip_risk_score', 'login_attempts_last_24h'],
      dtype='str')

In [228]:
y_train.keys()
# Our labelled dependent variable 

Index(['fraud_label'], dtype='str')

,user_id,Bangalore_prob,Chennai_prob,Delhi_prob,Hyderabad_prob,Mumbai_prob
0,U5261,0.0,0.0,0.0,0.0,0.0
1,U8927,0.0,0.0,0.0,0.0,0.0
2,U1965,0.0,0.0,0.0,0.0,0.0
3,U4533,0.0,0.0,0.0,0.0,0.0
4,U6077,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...
3999,U8501,0.0,0.0,0.0,0.0,0.0
4000,U8879,0.0,0.0,0.0,0.0,0.0
4001,U9494,0.0,0.0,0.0,0.0,0.0
4002,U2408,0.0,0.0,0.0,0.0,0.0


In [339]:
# Now we get the dummies for each location 
location_dummies = pd.get_dummies(x_train, columns = ['device_location'], dtype = int)
location_dummies = location_dummies[['user_id','device_location_Bangalore'	,
                                    'device_location_Chennai',	'device_location_Delhi',	
                                    'device_location_Hyderabad'	,'device_location_Mumbai']]


location_dummies = location_dummies.groupby(['user_id']).agg({'device_location_Bangalore': 'sum',
                                                              'device_location_Chennai': 'sum',
                                                              'device_location_Delhi': 'sum',
                                                              'device_location_Hyderabad': 'sum',
                                                              'device_location_Mumbai': 'sum'})

In [340]:
location_dummies[800:810]

,device_location_Bangalore,device_location_Chennai,device_location_Delhi,device_location_Hyderabad,device_location_Mumbai
user_id,,,,,
U2889,0,0,1,0,0
U2893,0,0,0,1,0
U2894,1,0,0,0,0
U2895,1,0,0,0,0
U2896,1,0,0,0,0
U2897,0,2,0,0,1
U2898,1,0,0,0,0
U2900,0,0,0,1,0
U2902,0,0,1,0,0


In [ ]:
bu

0       0.666667
1       0.833333
2       0.857143
3       0.833333
4       0.833333
          ...   
5245    0.875000
5246    0.833333
5247    0.666667
5248    0.857143
5249    0.833333
Name: location_rarity, Length: 5250, dtype: float64

,user_id,transaction_amount,device_location,account_age_days,transaction_hour,previous_failed_attempts,payment_mode,device_type,transaction_type,avg_transaction_amount,is_international,ip_risk_score,login_attempts_last_24h,Bangalore_prob,Chennai_prob,Delhi_prob,Hyderabad_prob,Mumbai_prob,rarity_location,location_rarity
0,U5261,29560.69,Bangalore,1803,9,1,NetBanking,iOS,Payment,29351.96,0,0.605,9,0.666667,0.833333,0.833333,0.833333,0.833333,Bangalore_prob,0.666667
1,U8927,2157.91,Hyderabad,1556,22,1,Card,iOS,Withdrawal,25085.59,0,0.174,6,0.833333,0.666667,0.833333,0.833333,0.833333,Hyderabad_prob,0.833333
2,U1965,13652.15,Delhi,1255,12,0,Wallet,Android,Payment,29771.50,0,0.205,7,0.714286,0.857143,0.857143,0.857143,0.714286,Delhi_prob,0.857143
3,U4533,4766.22,Hyderabad,1376,19,1,NetBanking,Web,Transfer,12047.71,0,0.345,2,0.833333,0.833333,0.833333,0.833333,0.666667,Hyderabad_prob,0.833333
4,U6077,33579.98,Mumbai,277,5,2,Card,Web,Payment,14137.25,0,0.299,3,0.666667,0.833333,0.833333,0.833333,0.833333,Mumbai_prob,0.833333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5245,U2408,7644.83,Bangalore,807,6,3,Wallet,Web,Transfer,3882.73,0,0.136,5,0.875000,0.750000,0.875000,0.750000,0.750000,Bangalore_prob,0.875000
5246,U3419,38187.62,Chennai,1843,23,1,NetBanking,iOS,Withdrawal,24719.16,0,0.751,7,0.833333,0.833333,0.833333,0.833333,0.666667,Chennai_prob,0.833333
5247,U3059,25472.50,Chennai,1753,11,3,UPI,Web,Withdrawal,9954.75,0,0.022,4,0.833333,0.666667,0.833333,0.833333,0.833333,Chennai_prob,0.666667
5248,U2739,29993.08,Mumbai,1964,9,0,Card,Android,Transfer,21533.14,0,0.991,2,0.857143,0.714286,0.714286,0.857143,0.857143,Mumbai_prob,0.857143


### PIPELINES
We will now experiment with making a scikit-learn pipeline.  
These are methods we can use to automate the data scince process by doing things like ETL, Pre-processing, Feature engineering and much more. 

We will create our own class to carry out transfromations on the train and test data separetely and then fit our model.   
This will inclove the use of:   
 - FunctionTransformer
 - One hot encoding (replcaing dummy variables in a previous iteration)
 - Standrad scaler so values like transaction amounts don't skew our model 

Let's get into it!!

In [13]:
# We mist start by imprting our pipeline functionality 
from sklearn.pipeline import Pipeline                   # creating a data pre-propcessing pipeline for our test set 
from sklearn.preprocessing import FunctionTransformer   # to create feature engineering functions 

In [ ]:
#Defining the class
from sklearn.base import TransformerMixin, BaseEstimator

class ETL(BaseEstimator, TransformerMixin): 
    def __init__(self, alpha = 1):
        self.alpha = alpha
        self.user_rarity_ = None       # user-level probabilities
        self.global_rarity_ = None     # global probabilities

    def fit(self,x , y = None):

        X = x.copy() 
        #Creating user level lookup table 
        # Let us start by getting making an empty dataframe we will use to record the prevalnce of each User ID device location 
        users_location_history = pd.DataFrame({ #we have an empty list for each users location history of transactions 
            "user_id" :[],
            "Bangalore_prob" :[],
            "Chennai_prob" :[],
            "Delhi_prob" :[],
            "Hyderabad_prob" :[],
            "Mumbai_prob" :[],
        })   
        users_location_history["user_id"] = X["user_id"].unique()     # this makes an empty dataframe of all of the users we have 
     

        columns = users_location_history[['Bangalore_prob','Chennai_prob','Delhi_prob','Hyderabad_prob','Mumbai_prob']] #columns where we want to assign zero to contents 
        col_array = np.array(columns)   #convert to array 

        nan_array = np.isnan(col_array) # finding all cells that are NaN in col_array
        col_array[nan_array] = 0        # if a cell in array col_arrau is NaN assign 0
        users_location_history[['Bangalore_prob','Chennai_prob','Delhi_prob','Hyderabad_prob','Mumbai_prob']] = col_array # now all the empty cells have been replaced with zeros          

        self.user_rarity_ = users_location_history
        # Now we get the dummies for each location 
        location_dummies = pd.get_dummies(X, columns = ['device_location'], dtype = int)
        location_dummies = location_dummies[['user_id','device_location_Bangalore'	,
                                            'device_location_Chennai',	'device_location_Delhi',	
                                            'device_location_Hyderabad'	,'device_location_Mumbai']]


        location_dummies = location_dummies.groupby(['user_id']).agg({'device_location_Bangalore': 'sum',
                                                                    'device_location_Chennai': 'sum',
                                                                    'device_location_Delhi': 'sum',
                                                                    'device_location_Hyderabad': 'sum',
                                                                    'device_location_Mumbai': 'sum'})

        col_array_prob = np.array(location_dummies[['device_location_Bangalore','device_location_Chennai',
                                                    'device_location_Delhi','device_location_Hyderabad',	
                                                    'device_location_Mumbai']])

        num_features = location_dummies.shape[1]
        total_per_user = col_array_prob.sum(axis = 1)
        alpha = 1
        prob_array = (1- (col_array_prob + alpha)/(total_per_user[:, None]+ alpha * num_features))
        users_location_history[['Bangalore_prob','Chennai_prob','Delhi_prob','Hyderabad_prob','Mumbai_prob']] = prob_array

        total_in_data = col_array_prob.sum(axis=0)
        total_across = col_array_prob.sum()
        total_prob_array = (1-(total_in_data + alpha)/ (total_across + alpha * num_features))
        total_prob_df = pd.DataFrame({
            "user_id" :["No_id"],
            "Bangalore_prob" :total_prob_array[0],
            "Chennai_prob" :total_prob_array[1],
            "Delhi_prob" :total_prob_array[2],
            "Hyderabad_prob" :total_prob_array[3],
            "Mumbai_prob" :total_prob_array[4],
        })

        lookup_value = pd.merge(X, users_location_history, on = "user_id", how = "left")
        lookup_value['rarity_location'] = lookup_value['device_location']+"_prob"
        
        col_index = {
            'Bangalore_prob' : 0,
            'Chennai_prob' :1,
            'Delhi_prob': 2,
            'Hyderabad_prob': 3,
            'Mumbai_prob' :4
        }
        prob_array_2 = lookup_value[["Bangalore_prob",
                    "Chennai_prob",
                    "Delhi_prob" ,
                    "Hyderabad_prob",
                    "Mumbai_prob"]].to_numpy()
        col_indices = lookup_value['rarity_location'].map(col_index).to_numpy()
        row_indices = np.arange(len(lookup_value))

        lookup_value['location_rarity'] = prob_array_2[row_indices,col_indices]
        self.location_rarity_log = lookup_value[['user_id','device_location','location_rarity']].copy()
        return lookup_value
    
    def drop_features(self, x):                           # drop features that we used to create our features like "user id"
        columns_to_drop = ['user_id','payment_mode', 'device_type', 'transaction_type','Bangalore_prob',	'Chennai_prob',	'Delhi_prob',	'Hyderabad_prob',	'Mumbai_prob',	'rarity_location' ]
        existing_columns = [ c for c in columns_to_drop if c in xl.columns]
        xl = xl.drop(columns = existing_columns)
        return xl

In [ ]:
    def location_tag(x):
        #creating the location buckets 
        fraud_location = {  
        "Mumbai":5,     # highest likelihood of fraud 
        "Hyderabad" :4, # second highest likelihood of fraud etc
        "Chennai" : 3,
        "Bangalore" : 2,
        "Delhi" : 1}
        x["fraud_bucket_location"] = x["device_location"].map(fraud_location)
        return x #returning our processed test dataframe 
 
    def risk_buckets(x):
        #creating our risk buckets 
        x["ip_risk_bucket"] = np.where((x['ip_risk_score'] < 0.51) | 
                                        (x['ip_risk_score'] > 0.86), 1, 0)        # assign a 1 if the risk is in the dangerous category and 0 itherwise 

        #creating the account age buckets 
        x["account_age_bucket"] = np.where((x['account_age_days'] >= 1190 ) &
                                            (x['account_age_days'] <= 1855), 1,0) # assign a 1 if we have an account in the most denagrous range and 0 otherwise
        return x
    
    def amount_averages(x):
        transaction_average = pd.DataFrame()                        # empty dataframe to store averages 

        transaction_average["user_id"] = x['user_id'].unique()      # this represents each users average transaction amount for this round of transactions 

        means = x.groupby(['user_id'])['transaction_amount'].mean() # get the mean of each transaction amount from the test data for each User ID

        transaction_average["days_average"] = transaction_average["user_id"].map(means) # add this mean called "days_average" to the dataframe 

        averages_per_user = x.groupby(['user_id'])["avg_transaction_amount"].unique()   # conduct the same procedure for the historic known average of User IDs

        typical_average = averages_per_user.apply(np.mean)                              # get the mean of each average since some User IDs have multiple historic averages 

        transaction_average["typical_average"] = transaction_average["user_id"].map(typical_average) # add this updated historic average to the dataframe 
        x = x.merge(transaction_average, on = "user_id", how = 'left')                               # add this dataframe to x using joins and duplicate some for each User ID so left join
        return x
    
    def transaction_outliers(x):
        days_sd = np.std(x['days_average']) # standard deviation for the days average column (transaction amounts follow roughly uniform distribution form the EDA)

        outlier_days_avg = pd.DataFrame()   # empty dataframe to get the outlier transactions 

        outlier_days_avg["outlier_days_average"] = np.where( (x['transaction_amount'] <= x['days_average'] -  2.5* days_sd) | 
            (x['transaction_amount'] >= x['days_average'] + 2.5 * days_sd), 1,0)           #outliers on days average are transaction amounts outside 2.5 standard deviations from the mean
        
        typical_sd = np.std(x['typical_average']) # similar to the above 

        outlier_typical_avg = pd.DataFrame()
        outlier_typical_avg["outlier_typical_average"] = np.where( (x['transaction_amount'] <= x['days_average'] -  2.5* typical_sd) | 
            (x['transaction_amount'] >= x['days_average'] + 2.5 * typical_sd), 1,0)         # outliers outside 2 standard deviations from the mean are flagged with 1 otherwise 0
        
        x = pd.concat([x,outlier_days_avg,outlier_typical_avg ], axis =1)                   # adding the outliers in terms of the aveage amounts to the test dataframe x
        return x
    
    def international_outliers(x):
        commom_international = x.groupby('user_id')['is_international'].agg(lambda x: pd.Series.mode(x)[0]) # get the most common location (domestric or international) where each User does transactions from
        x['transaction_usual_international'] = x["user_id"].map(commom_international)                       # add this mode to the test dataframe
        x["unexpected_location"] = np.where((x['is_international'] == x['transaction_usual_international'] ) , 0,1) # assign 0 if the transaction is in the usual location and 1 otherwise
        return x

    def login_outliers(x):
        usual_login_attempts = x.groupby('user_id')['previous_failed_attempts'].mean()          # mean of the number of number of attempts to login by User ID 
        x['usual_login_attempts'] = x["user_id"].map(usual_login_attempts)                      # map these onto the test dataframe 
        x["unexpected_login_attempts"] = np.where((x['previous_failed_attempts'] <= x['usual_login_attempts'] ) , 0,1) #assign 0 is the transactions attempts are less than or equal to the normla amount and 1 otherwise 
        return x

In [511]:
#scaling and get dummies (one hot encoding our variables)
numeric_features = ['transaction_amount', 'account_age_days',
                     'transaction_hour', 'ip_risk_score', ] 
    
categorical_features = ['device_location', 'previous_failed_attempts',  'is_international', 'login_attempts_last_24h']

In [512]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression


numeric_transformation = StandardScaler()
categorical_transformation = OneHotEncoder()

preprocess_data = ColumnTransformer(
    transformers= ([
        ("num" , numeric_transformation, numeric_features),
        ("cat", categorical_transformation,categorical_features)
    ]))

In [513]:
etl = ETL(None)
locationtransformer = FunctionTransformer(lambda X: etl.location_proba(X), validate=False)
droptransformer = FunctionTransformer(lambda X: etl.drop_features(X), validate= False)


pipeline = Pipeline([
    ('location_rarity', locationtransformer),
    ("drop_features", droptransformer),
    ("preprocess", preprocess_data),
    ("Logistic_regression", LogisticRegression())
])

## MODELLING 

In [514]:
#We will now us the pipleine on our test data 
pipeline.fit(x_train, y_train)

<class 'pandas.DataFrame'>


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/utils/validation.py:1352: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('location_rarity', ...), ('drop_features', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"func func: callable, default=NoneThe callable to use for the transformation. This will be passedthe same arguments as transform, with args and kwargs forwarded.If func is None, then func will be the identity function.",<function <la...t 0x1225e4f60>
,"inverse_func inverse_func: callable, default=NoneThe callable to use for the inverse transformation. This will bepassed the same arguments as inverse transform, with args andkwargs forwarded. If inverse_func is None, then inverse_funcwill be the identity function.",None
,"validate validate: bool, default=FalseIndicate that the input X array should be checked before calling``func``. The possibilities are:- If False, there is no input validation.- If True, then X will be converted to a 2-dimensional NumPy array or sparse matrix. If the conversion is not possible an exception is raised... versionchanged:: 0.22 The default of ``validate`` changed from True to False.",False
,"accept_sparse accept_sparse: bool, default=FalseIndicate that func accepts a sparse matrix as input. If validate isFalse, this has no effect. Otherwise, if accept_sparse is false,sparse matrix inputs will cause an exception to be raised.",False
,"check_inverse check_inverse: bool, default=TrueWhether to check that or ``func`` followed by ``inverse_func`` leads tothe original inputs. It can be used for a sanity check, raising awarning when the condition is not fulfilled... versionadded:: 0.20",True
,"feature_names_out feature_names_out: callable, 'one-to-one' or None, default=NoneDetermines the list of feature names that will be returned by the`get_feature_names_out` method. If it is 'one-to-one', then the outputfeature names will be equal to the input feature names. If it is acallable, then it must take two positional arguments: this`FunctionTransformer` (`self`) and an array-like of input feature names(`input_features`). It must return an array-like of output featurenames. The `get_feature_names_out` method is only defined if`feature_names_out` is not None.See ``get_feature_names_out`` for more details... versionadded:: 1.1",None
,"kw_args kw_args: dict, default=NoneDictionary of additional keyword ar

In [515]:
print(etl.location_rarity_log.shape)

(5250, 3)


## EVALUATION AND TESTING 
We have trained our model on the train set.   
We can now use the metrics mentined earlier like balanced accuracy, precision and recall to judge our models

**Precision** - of all the positievs identified how many were truly positive?  
**Recall**- of all possible positive instances, how many did the model catch? 

For fraud detection, we value recall as our main metric.   
This is beacuse catching every fraudlent trasaction allows for the least damage to our client base as opposed to flagging a genuine transaction as fraud that can be reversed wuth no harm to the user. 

In [516]:
y_pred_log = pipeline.predict(x_test) #making predictions on the train data 


<class 'pandas.DataFrame'>


In [517]:
# Importing balanced accuracy
from sklearn.metrics import accuracy_score,balanced_accuracy_score, precision_recall_curve, confusion_matrix,precision_score, recall_score, ConfusionMatrixDisplay #importing all necesarry metrics

In [518]:
accuracy_score(y_test, y_pred_log) #we have a very high accuracy score of 0.935


0.9346666666666666

In [519]:
balanced_accuracy_score(y_test,y_pred_log) #our balaned accuracy score is very lowe at 0.5

0.5

In [520]:
precision_score(y_test,y_pred_log) # precision is 0%

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


0.0

In [521]:
recall_score(y_test,y_pred_log) #recall is 0%

0.0

Okay, there are many takeaways from this experiment:   

1) Our Logistic regression model is very aggressive. It is able to catch all of the fraud cases at 100% recall (which is great) but it does so by guessing everything as fraud. 
- This means that every custimer would have every transactions falgged as fraud if this model was implemented which is a horrendous user expirience. 
- As a result the balanced accuracy falls since we got 100% of 50% and 0% of 50% of predcitions correct. 

2) Our Random Forest is much better with a recall at 70.7.0% so it catches the majority of fraud. 
-  However we still have many majority of non fraud cases being calssified as fraud so it will still be very inconveneint for all of our users.   

**All 3 models perform relatively poorly in atleast 2 metrics that are the most important.**. 

Therefore we can use resampling techniques to solve this issue.   
For severly unbalanced data, there is a resampling technique called SMOTE resampling. 

### SMOTE 
Synthetic Minority Over-Sampling Technique (SMOTE)

SMOTE is a data-level resampling technique that generates synthetic (artificial) samples for the minority class.   
Instead of simply duplicating existing examples, it creates new data points by interpolating between existing ones.   
This approach allows the model to learn broader patterns and reduces the risk of overfitting to repeated samples

In [ ]:
#First let us start by importing the necessary library 
from imblearn.over_sampling import SMOTE 
from imblearn.pipeline import Pipeline as Pipe

In [ ]:
# Now we will resample our data 
# This will involve adding the resampling step midway through into our pipeline 

featuretransformer_SMOTE = FunctionTransformer(new_features)


smt = SMOTE(sampling_strategy="minority", random_state= 40)

pipeline6 = Pipe([
    ("location_buckets", locationtransformer),
    ("risk_buckets", riskransformer),
    ("amount_average_outliers", amount_avgtransformer),
    ("transaction_outliers", transaction_outliertransformer),
    ("international_outliers", international_outliertransformer),
    ("login_attempts_outlier", login_outliertransformer,),
    ("drop_features", droptransformer),
    ("preprocess", preprocess_data),
     ("smote", smt),
    ("Logistic_regression", LogisticRegression(class_weight= {0:1, 1:1.20}))
])

pipeline7 = Pipe([
    ("feature_engineering", featuretransformer_SMOTE),
    ("preprocess", preprocess_data),
    ("smote", smt),
    ("Random Forest", RandomForestClassifier(class_weight= "balanced"))
])

pipeline8 = Pipe([
    ("feature_engineering", featuretransformer_SMOTE),
    ("preprocess", preprocess_data),
    ("smote", smt),
    ("Gardient Boosting", GradientBoostingClassifier(n_estimators= 500,
                                                     max_depth= 3,
                                                     min_samples_split= 2,
                                                     learning_rate= 0.03))
])